Diese Analyse prüft den Datensatz, aus dem der StudiScore entsteht: **amenity-Nodes** aus
OpenStreetMap für die ganze Schweiz. Leitfragen:

1. Wie viele Orte gibt es pro Kategorie, und wie vollständig sind die Tags?
2. Lässt sich ein Ort über `addr:postcode` einer PLZ zuordnen, oder braucht es die Umkreissuche?
3. Wie sieht ein erster Score für PLZ **8400 Winterthur** aus?

Die Daten lädt `src/fetch_osm.py` über die Overpass API nach `data/raw/osm-output.json`.

In [1]:
import json
import sys
from pathlib import Path

import folium
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from categories import AMENITY_TO_CATEGORY, CATEGORIES

# Feste Farbreihenfolge pro Kategorie (Farbe folgt der Kategorie, nicht dem Rang)
COLORS = dict(zip(CATEGORIES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
CAT_ORDER = list(CATEGORIES)
px.defaults.template = "plotly_white"
pio.renderers.default = "plotly_mimetype+notebook_connected"  # HTML-Output, damit Quarto die Grafiken zeigt

## Daten laden

In [2]:
raw = json.loads((ROOT / "data" / "raw" / "osm-output.json").read_text())
print("OSM-Stand:", raw["osm3s"]["timestamp_osm_base"])

df = pd.DataFrame(
    {
        "id": el["id"],
        "lat": el["lat"],
        "lon": el["lon"],
        "amenity": el["tags"].get("amenity"),
        "name": el["tags"].get("name"),
        "postcode": el["tags"].get("addr:postcode"),
        "city": el["tags"].get("addr:city"),
        "opening_hours": el["tags"].get("opening_hours"),
        "website": el["tags"].get("website") or el["tags"].get("contact:website"),
        "wheelchair": el["tags"].get("wheelchair"),
        "n_tags": len(el["tags"]),
    }
    for el in raw["elements"]
)
df["category"] = df["amenity"].map(AMENITY_TO_CATEGORY)
df["category"] = pd.Categorical(df["category"], categories=CAT_ORDER, ordered=True)
print(f"{len(df):,} Nodes, {df['id'].duplicated().sum()} doppelte IDs")
df.head()

OSM-Stand: 2026-10-22T06:37:48Z
41,241 Nodes, 0 doppelte IDs


,id,lat,lon,amenity,name,postcode,city,opening_hours,website,wheelchair,n_tags,category
0,1014954,46.492849,7.562856,cinema,Ciné Rex,NaN,NaN,NaN,https://www.cine-rex.ch/kino.shtml,NaN,7,Kultur & Freizeit
1,8096510,46.729090,8.447001,pub,Susten Kulm,NaN,NaN,NaN,NaN,NaN,5,Nachtleben
2,8204263,46.577370,8.387538,cafe,Snack-Buvette Belvédère,NaN,NaN,NaN,http://www.gletscher.ch/snack-buvette-belvedere/,yes,6,Essen & Kaffee
3,9332247,46.684532,7.853794,fast_food,McDonald's,NaN,NaN,NaN,NaN,NaN,11,Essen & Kaffee
4,18476816,47.044597,7.505473,restaurant,Gourmet Party Service,3303,NaN,NaN,NaN,yes,6,Essen & Kaffee


## Wie viele Orte pro Kategorie?

In [3]:
per_cat = df["category"].value_counts().reindex(CAT_ORDER).rename_axis("Kategorie").reset_index(name="Anzahl")
fig = px.bar(per_cat, x="Anzahl", y="Kategorie", orientation="h", color="Kategorie",
             color_discrete_map=COLORS, text="Anzahl")
fig.update_traces(texttemplate="%{text:,}", textposition="outside", hovertemplate="%{y}: %{x:,} Orte<extra></extra>")
fig.update_layout(showlegend=False, yaxis_title=None, xaxis_title="Anzahl Nodes", height=360)
fig.show()

In [4]:
per_amenity = (df.groupby(["category", "amenity"], observed=True).size()
                 .reset_index(name="Anzahl").sort_values("Anzahl", ascending=True))
fig = px.bar(per_amenity, x="Anzahl", y="amenity", color="category", orientation="h",
             color_discrete_map=COLORS, category_orders={"category": CAT_ORDER},
             labels={"category": "Kategorie", "amenity": ""})
fig.update_layout(height=700, xaxis_type="log", xaxis_title="Anzahl Nodes (log)")
fig.show()

**Beobachtung:** Die Kategorien sind sehr ungleich gross. *Essen & Kaffee* stellt mehr als
die Hälfte aller Nodes (allein ~15'500 `restaurant`), *Lernen* hat nur ~1'400. Rohe Zählwerte lassen sich
deshalb nicht direkt addieren: jede Kategorie muss für den Score separat normiert werden.

## Datenqualität: Wie vollständig sind die Tags?

In [5]:
fields = ["name", "postcode", "city", "opening_hours", "website", "wheelchair"]
completeness = (df.groupby("category", observed=True)[fields]
                  .agg(lambda s: s.notna().mean() * 100).round(1))
fig = px.imshow(completeness, text_auto=".0f", color_continuous_scale="Blues", zmin=0, zmax=100,
                labels=dict(x="Tag", y="Kategorie", color="% vorhanden"), aspect="auto")
fig.update_layout(height=380)
fig.show()

In [6]:
completeness.assign(Gesamt_n=df["category"].value_counts())

,name,postcode,city,opening_hours,website,wheelchair,Gesamt_n
category,,,,,,,
Lernen,97.9,71.3,71.7,35.3,79.7,15.8,1426
Essen & Kaffee,97.6,41.1,37.6,49.6,48.4,24.5,22394
Nachtleben,96.6,36.2,33.6,39.1,35.4,23.6,3411
Kultur & Freizeit,97.4,44.0,42.0,12.6,60.3,20.7,1534
Mobilität,87.0,38.8,38.8,57.6,68.1,13.0,2853
Alltag & Gesundheit,77.0,39.5,38.3,39.8,31.6,22.9,9623


**Beobachtung:** `name` ist bei ~97 % vorhanden (Ausnahme *Alltag*: Bankomaten haben oft keinen
Namen). `addr:postcode` fehlt dagegen bei rund 60 % der Nodes, nur *Lernen* liegt bei ~70 %. Wer nur über die PLZ-Tags filtert, verliert viele Orte. Das spricht für den
geplanten Ansatz im technischen Konzept: **GeoJSON + `2dsphere`-Index in MongoDB** und
eine Umkreissuche um den PLZ-Mittelpunkt.

## PLZ-Tags genauer

In [7]:
pc = df["postcode"].dropna().str.strip()
valid = pc.str.fullmatch(r"\d{4}")
print(f"Nodes mit addr:postcode: {len(pc):,} von {len(df):,} ({len(pc)/len(df):.0%})")
print(f"davon gültige 4-stellige CH-PLZ: {valid.mean():.1%}")
print("Beispiele ungültig:", pc[~valid].unique()[:10].tolist())
print(f"Anzahl verschiedener PLZ: {pc[valid].nunique():,}")

Nodes mit addr:postcode: 17,047 von 41,241 (41%)
davon gültige 4-stellige CH-PLZ: 100.0%
Beispiele ungültig: []
Anzahl verschiedener PLZ: 2,062


In [8]:
top = (df[df["postcode"].str.fullmatch(r"\d{4}", na=False)]
         .groupby(["postcode", "category"], observed=True).size().reset_index(name="Anzahl"))
top_pc = top.groupby("postcode")["Anzahl"].sum().nlargest(15).index
fig = px.bar(top[top["postcode"].isin(top_pc)], x="postcode", y="Anzahl", color="category",
             color_discrete_map=COLORS, category_orders={"category": CAT_ORDER, "postcode": list(top_pc)},
             labels={"postcode": "PLZ", "category": "Kategorie"})
fig.update_layout(title="Top 15 PLZ nach Anzahl getaggter Orte", height=420)
fig.show()

## Räumliche Verteilung

In [9]:
fig = px.density_map(df, lat="lat", lon="lon", radius=4, zoom=6.6,
                     center={"lat": 46.8, "lon": 8.2}, map_style="carto-positron",
                     color_continuous_scale="Blues")
fig.update_layout(height=520, margin=dict(l=0, r=0, t=0, b=0), coloraxis_showscale=False)
fig.show()

Die Orte konzentrieren sich im Mittelland und in den Städten (Zürich, Genf, Basel, Bern,
Lausanne). Ländliche und alpine PLZ werden im Score entsprechend tief liegen. Das ist
gewollt, aber für die Interpretation wichtig.

## Fallbeispiel: 8400 Winterthur

In [10]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# Mittelpunkt einer PLZ = Median der Koordinaten aller Nodes mit diesem PLZ-Tag
tagged = df[df["postcode"].str.fullmatch(r"\d{4}", na=False)]
centroids = tagged.groupby("postcode").agg(lat=("lat", "median"), lon=("lon", "median"), n=("id", "size"))
RADIUS_KM = 1.5

c = centroids.loc["8400"]
df["dist_8400"] = haversine_km(c["lat"], c["lon"], df["lat"], df["lon"])
near = df[df["dist_8400"] <= RADIUS_KM]
print(f"Mittelpunkt 8400: {c['lat']:.4f}, {c['lon']:.4f}")
print(f"{len(near)} Orte im Umkreis von {RADIUS_KM} km, davon {near['postcode'].eq('8400').sum()} mit PLZ-Tag 8400")
near["category"].value_counts().reindex(CAT_ORDER)

Mittelpunkt 8400: 47.4993, 8.7266
427 Orte im Umkreis von 1.5 km, davon 249 mit PLZ-Tag 8400


category
Lernen                  12
Essen & Kaffee         214
Nachtleben              48
Kultur & Freizeit       22
Mobilität               27
Alltag & Gesundheit    104
Name: count, dtype: int64

## Score-Prototyp

Erster Entwurf der Score-Logik, um zu sehen, ob die Daten trennscharf sind:

1. Für jede PLZ alle Orte im Umkreis von 1.5 km um den PLZ-Mittelpunkt zählen.
2. Pro Kategorie normieren: Anzahl / 95. Perzentil über alle PLZ, gekappt bei 1.
3. Gewichtet summieren und auf 0–100 skalieren.

Die Gewichte sind **Platzhalter** und müssen im Team noch festgelegt werden.

In [12]:
WEIGHTS = {"Lernen": 0.25, "Essen & Kaffee": 0.20, "Nachtleben": 0.15,
           "Kultur & Freizeit": 0.15, "Mobilität": 0.10, "Alltag & Gesundheit": 0.15}

lat, lon, cat = df["lat"].to_numpy(), df["lon"].to_numpy(), df["category"].astype(str).to_numpy()
rows = []
for plz, p in centroids[centroids["n"] >= 3].iterrows():
    mask = haversine_km(p["lat"], p["lon"], lat, lon) <= RADIUS_KM
    counts = pd.Series(cat[mask]).value_counts()
    rows.append({"plz": plz, **{k: int(counts.get(k, 0)) for k in CAT_ORDER}})
counts = pd.DataFrame(rows).set_index("plz")

cap = counts.quantile(0.95).clip(lower=1)
norm = (counts / cap).clip(upper=1)
score = (norm * pd.Series(WEIGHTS)).sum(axis=1) * 100
result = counts.assign(score=score.round(0)).sort_values("score", ascending=False)
print(f"Score für {len(result):,} PLZ berechnet")
print("95%-Kappung pro Kategorie:", cap.round(0).to_dict())

Score für 1,042 PLZ berechnet
95%-Kappung pro Kategorie: {'Lernen': 13.0, 'Essen & Kaffee': 243.0, 'Nachtleben': 42.0, 'Kultur & Freizeit': 20.0, 'Mobilität': 31.0, 'Alltag & Gesundheit': 74.0}


In [15]:
result.head(15)

,Lernen,Essen & Kaffee,Nachtleben,Kultur & Freizeit,Mobilität,Alltag & Gesundheit,score
plz,,,,,,,
3007,61,370,74,35,123,120,100.0
3001,77,440,91,44,150,130,100.0
8001,49,958,221,67,98,217,100.0
3000,90,435,83,40,142,131,100.0
4057,21,377,85,29,31,97,100.0
8002,26,745,159,52,80,178,100.0
8032,27,420,72,30,56,114,100.0
8003,22,613,167,47,90,109,100.0
8004,36,982,248,72,115,216,100.0
